In [5]:
# ============================================================
# MNIST HANDWRITTEN DIGIT CLASSIFICATION
#
# Activities 1, 2 and 3 using GridSearchCV
#
# Activity 1:
#   Optimise activation function
#
# Activity 2:
#   Optimise activation function + optimiser
#
# Activity 3:
#   Optimise activation function + optimiser + neurons
#
# Final model:
#   Retrain the best Activity 3 configuration on all 60,000
#   MNIST training images and evaluate on the 10,000 test images.
#
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import time
import random
import os

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from scikeras.wrappers import KerasClassifier


# ============================================================
# 2. SETTINGS
# ============================================================

SEED = 42

NUM_CLASSES = 10
INPUT_SHAPE = (28, 28, 1)

BATCH_SIZE = 128
EPOCHS = 2

# Use 2 while testing.
# Change to 3 for the final coursework experiment.
CV_FOLDS = 2

DROPOUT_RATE = 0.5

# Activity 1
ACTIVATIONS = [
    "relu",
    "elu",
    "sigmoid",
    "tanh"
]

# Activity 2
OPTIMISERS = [
    "adam",
    "sgd",
    "rmsprop",
    "adagrad"
]

# Activity 3
NEURONS = [
    16,
    32,
    64,
    128,
    256
]


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 4. LOAD MNIST DATASET
# ============================================================

print("\nLoading MNIST dataset...")

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()


# ============================================================
# 5. NORMALISE IMAGES
# ============================================================

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0


# Add channel dimension:
# (60000, 28, 28) -> (60000, 28, 28, 1)

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)


# Keep integer labels for GridSearchCV / SciKeras
y_train_original = y_train.copy()
y_test_original = y_test.copy()


# ============================================================
# 6. CREATE TRAINING / VALIDATION SPLIT
# ============================================================

x_train_main, x_val, y_train_main, y_val = train_test_split(
    x_train,
    y_train_original,
    test_size=0.1,
    random_state=SEED,
    stratify=y_train_original
)


print("\nDataset sizes:")
print("Training:", x_train_main.shape[0])
print("Validation:", x_val.shape[0])
print("Test:", x_test.shape[0])


# ============================================================
# 7. BUILD KERAS MODEL
# ============================================================

def build_model(
    activation="relu",
    optimiser="adam",
    neurons=10,
    dropout_rate=0.5
):

    model = keras.Sequential([

        keras.Input(shape=INPUT_SHAPE),

        # Convolutional layer 1
        layers.Conv2D(
            32,
            kernel_size=(3, 3),
            activation="relu"
        ),

        # Pooling
        layers.MaxPooling2D(
            pool_size=(2, 2)
        ),

        # Convolutional layer 2
        layers.Conv2D(
            64,
            kernel_size=(3, 3),
            activation="relu"
        ),

        # Pooling
        layers.MaxPooling2D(
            pool_size=(2, 2)
        ),

        # Flatten
        layers.Flatten(),

        # Dropout
        layers.Dropout(dropout_rate),

        # Hidden dense layer being optimised
        layers.Dense(
            neurons,
            activation=activation
        ),

        # Output layer
        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ])


    # --------------------------------------------------------
    # Select optimiser
    # --------------------------------------------------------

    if optimiser == "adam":

        opt = keras.optimizers.Adam()

    elif optimiser == "sgd":

        opt = keras.optimizers.SGD()

    elif optimiser == "rmsprop":

        opt = keras.optimizers.RMSprop()

    elif optimiser == "adagrad":

        opt = keras.optimizers.Adagrad()

    else:

        raise ValueError(
            f"Unknown optimiser: {optimiser}"
        )


    # --------------------------------------------------------
    # Compile
    # --------------------------------------------------------

    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=opt,
        metrics=["accuracy"]
    )

    return model


# ============================================================
# 8. CREATE SCIKERAS CLASSIFIER
# ============================================================

keras_classifier = KerasClassifier(
    model=build_model,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,
    random_state=SEED
)


# ============================================================
# 9. HELPER FUNCTION
# ============================================================

def evaluate_best_model(
    best_model,
    x_validation,
    y_validation,
    activity_name
):

    print("\n" + "=" * 70)
    print(activity_name)
    print("=" * 70)

    start_time = time.time()

    predictions = best_model.predict(x_validation)

    evaluation_time = time.time() - start_time

    accuracy = accuracy_score(
        y_validation,
        predictions
    )

    precision = precision_score(
        y_validation,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        predictions,
        average="weighted",
        zero_division=0
    )

    print(
        f"Validation accuracy: {accuracy:.6f} "
        f"({accuracy * 100:.2f}%)"
    )

    print(f"Precision: {precision:.6f}")
    print(f"Recall:    {recall:.6f}")
    print(f"F1 score:  {f1:.6f}")

    return {
        "validation_accuracy": accuracy,
        "validation_precision": precision,
        "validation_recall": recall,
        "validation_f1": f1,
        "evaluation_time_seconds": evaluation_time
    }


# ============================================================
# 10. ACTIVITY 1
#
# Find the best activation function.
#
# Optimiser = Adam
# Neurons = 10
# ============================================================

print("\n\n")
print("#" * 70)
print("ACTIVITY 1 - ACTIVATION FUNCTION GRID SEARCH")
print("#" * 70)

start_activity1 = time.time()


param_grid_activity1 = {

    "model__activation": ACTIVATIONS,

    "model__optimiser": ["adam"],

    "model__neurons": [10]

}


grid_activity1 = GridSearchCV(

    estimator=keras_classifier,

    param_grid=param_grid_activity1,

    cv=CV_FOLDS,

    scoring="accuracy",

    verbose=2,

    n_jobs=1,

    return_train_score=True

)


grid_activity1.fit(
    x_train_main,
    y_train_main
)


activity1_time = time.time() - start_activity1


print("\nActivity 1 complete.")

print(
    "\nBest parameters:"
)

print(
    grid_activity1.best_params_
)

print(
    f"\nBest cross-validation accuracy: "
    f"{grid_activity1.best_score_:.6f}"
)

print(
    f"Activity 1 time: "
    f"{activity1_time:.2f} seconds"
)


# Evaluate best model on separate validation set

activity1_validation = evaluate_best_model(
    grid_activity1.best_estimator_,
    x_val,
    y_val,
    "Activity 1 Validation Results"
)


# Save GridSearch results

activity1_results = pd.DataFrame(
    grid_activity1.cv_results_
)

activity1_results.to_csv(
    "activity1_gridsearch_results.csv",
    index=False
)


# ============================================================
# 11. ACTIVITY 2
#
# Find the best activation + optimiser combination.
#
# Neurons = 10
# ============================================================

print("\n\n")
print("#" * 70)
print("ACTIVITY 2 - ACTIVATION + OPTIMISER GRID SEARCH")
print("#" * 70)

start_activity2 = time.time()


param_grid_activity2 = {

    "model__activation": ACTIVATIONS,

    "model__optimiser": OPTIMISERS,

    "model__neurons": [10]

}


grid_activity2 = GridSearchCV(

    estimator=keras_classifier,

    param_grid=param_grid_activity2,

    cv=CV_FOLDS,

    scoring="accuracy",

    verbose=2,

    n_jobs=1,

    return_train_score=True

)


grid_activity2.fit(
    x_train_main,
    y_train_main
)


activity2_time = time.time() - start_activity2


print("\nActivity 2 complete.")

print(
    "\nBest parameters:"
)

print(
    grid_activity2.best_params_
)

print(
    f"\nBest cross-validation accuracy: "
    f"{grid_activity2.best_score_:.6f}"
)

print(
    f"Activity 2 time: "
    f"{activity2_time:.2f} seconds"
)


# Evaluate best model

activity2_validation = evaluate_best_model(
    grid_activity2.best_estimator_,
    x_val,
    y_val,
    "Activity 2 Validation Results"
)


# Save results

activity2_results = pd.DataFrame(
    grid_activity2.cv_results_
)

activity2_results.to_csv(
    "activity2_gridsearch_results.csv",
    index=False
)


# ============================================================
# 12. ACTIVITY 3
#
# Find the best:
#
# activation + optimiser + neurons
# ============================================================

print("\n\n")
print("#" * 70)
print("ACTIVITY 3 - FULL GRID SEARCH")
print("#" * 70)

print(
    "\nNumber of combinations:",
    len(ACTIVATIONS)
    * len(OPTIMISERS)
    * len(NEURONS)
)

print(
    "Cross-validation folds:",
    CV_FOLDS
)

print(
    "Total model fits:",
    len(ACTIVATIONS)
    * len(OPTIMISERS)
    * len(NEURONS)
    * CV_FOLDS
)


start_activity3 = time.time()


param_grid_activity3 = {

    "model__activation": ACTIVATIONS,

    "model__optimiser": OPTIMISERS,

    "model__neurons": NEURONS

}


grid_activity3 = GridSearchCV(

    estimator=keras_classifier,

    param_grid=param_grid_activity3,

    cv=CV_FOLDS,

    scoring="accuracy",

    verbose=2,

    n_jobs=1,

    return_train_score=True

)


grid_activity3.fit(
    x_train_main,
    y_train_main
)


activity3_time = time.time() - start_activity3


print("\nActivity 3 complete.")

print(
    "\nBest parameters:"
)

print(
    grid_activity3.best_params_
)

print(
    f"\nBest cross-validation accuracy: "
    f"{grid_activity3.best_score_:.6f}"
)

print(
    f"Activity 3 time: "
    f"{activity3_time:.2f} seconds"
)


# Evaluate best Activity 3 model

activity3_validation = evaluate_best_model(
    grid_activity3.best_estimator_,
    x_val,
    y_val,
    "Activity 3 Validation Results"
)


# Save complete GridSearch results

activity3_results = pd.DataFrame(
    grid_activity3.cv_results_
)

activity3_results.to_csv(
    "activity3_gridsearch_results.csv",
    index=False
)


# ============================================================
# 13. DISPLAY TOP ACTIVITY 3 CONFIGURATIONS
# ============================================================

print("\n")
print("=" * 70)
print("TOP ACTIVITY 3 CONFIGURATIONS")
print("=" * 70)


top_activity3 = activity3_results.sort_values(
    "rank_test_score"
).head(10)


print(
    top_activity3[
        [
            "param_model__activation",
            "param_model__optimiser",
            "param_model__neurons",
            "mean_test_score",
            "std_test_score",
            "rank_test_score"
        ]
    ].to_string(index=False)
)


# ============================================================
# 14. SELECT BEST CONFIGURATION
# ============================================================

best_params = grid_activity3.best_params_

best_activation = best_params[
    "model__activation"
]

best_optimiser = best_params[
    "model__optimiser"
]

best_neurons = best_params[
    "model__neurons"
]


print("\n")
print("=" * 70)
print("SELECTED FINAL CONFIGURATION")
print("=" * 70)

print(
    "Activation:",
    best_activation
)

print(
    "Optimiser:",
    best_optimiser
)

print(
    "Neurons:",
    best_neurons
)

print(
    f"CV accuracy: "
    f"{grid_activity3.best_score_:.6f}"
)


# ============================================================
# 15. RETRAIN FINAL MODEL ON ALL 60,000 TRAINING IMAGES
# ============================================================

print("\n\n")
print("#" * 70)
print("FINAL MODEL TRAINING")
print("#" * 70)

print(
    "\nRetraining best configuration on all "
    "60,000 training images..."
)


start_final_training = time.time()


final_model = build_model(

    activation=best_activation,

    optimiser=best_optimiser,

    neurons=best_neurons,

    dropout_rate=DROPOUT_RATE

)


final_history = final_model.fit(

    x_train,

    y_train_original,

    batch_size=BATCH_SIZE,

    epochs=EPOCHS,

    verbose=1

)


final_training_time = (
    time.time() - start_final_training
)


# ============================================================
# 16. FINAL TEST EVALUATION
# ============================================================

print("\n")
print("#" * 70)
print("FINAL TEST EVALUATION")
print("#" * 70)


test_loss, test_accuracy = final_model.evaluate(
    x_test,
    y_test_original,
    verbose=0
)


# Predictions

y_test_predictions = final_model.predict(
    x_test,
    verbose=0
)


y_test_predictions = np.argmax(
    y_test_predictions,
    axis=1
)


# Metrics

test_precision = precision_score(
    y_test_original,
    y_test_predictions,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    y_test_original,
    y_test_predictions,
    average="weighted",
    zero_division=0
)

test_f1 = f1_score(
    y_test_original,
    y_test_predictions,
    average="weighted",
    zero_division=0
)


# ============================================================
# 17. PRINT FINAL RESULTS
# ============================================================

print("\nFinal model:")
print(
    f"Activation: {best_activation}"
)

print(
    f"Optimiser: {best_optimiser}"
)

print(
    f"Neurons: {best_neurons}"
)

print(
    f"\nTest loss: {test_loss:.6f}"
)

print(
    f"Test accuracy: {test_accuracy:.6f} "
    f"({test_accuracy * 100:.2f}%)"
)

print(
    f"Precision: {test_precision:.6f}"
)

print(
    f"Recall: {test_recall:.6f}"
)

print(
    f"F1 score: {test_f1:.6f}"
)

print(
    f"Training time: "
    f"{final_training_time:.2f} seconds"
)


# ============================================================
# 18. CLASSIFICATION REPORT
# ============================================================

print("\n")
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)


report = classification_report(
    y_test_original,
    y_test_predictions,
    digits=4
)

print(report)


# ============================================================
# 19. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test_original,
    y_test_predictions
)


print("\n")
print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(cm)


# ============================================================
# 20. SAVE FINAL RESULTS
# ============================================================

final_results = pd.DataFrame({

    "activation": [best_activation],

    "optimiser": [best_optimiser],

    "neurons": [best_neurons],

    "cv_accuracy": [
        grid_activity3.best_score_
    ],

    "test_loss": [
        test_loss
    ],

    "test_accuracy": [
        test_accuracy
    ],

    "test_precision": [
        test_precision
    ],

    "test_recall": [
        test_recall
    ],

    "test_f1": [
        test_f1
    ],

    "training_time_seconds": [
        final_training_time
    ]

})


final_results.to_csv(
    "final_model_results.csv",
    index=False
)


# ============================================================
# 21. SAVE CONFUSION MATRIX
# ============================================================

cm_dataframe = pd.DataFrame(
    cm,
    index=[
        f"Actual_{i}"
        for i in range(NUM_CLASSES)
    ],
    columns=[
        f"Predicted_{i}"
        for i in range(NUM_CLASSES)
    ]
)


cm_dataframe.to_csv(
    "confusion_matrix.csv"
)


# ============================================================
# 22. SAVE TRAINING HISTORY
# ============================================================

history_dataframe = pd.DataFrame(
    final_history.history
)

history_dataframe.insert(
    0,
    "epoch",
    range(
        1,
        len(history_dataframe) + 1
    )
)


history_dataframe.to_csv(
    "final_training_history.csv",
    index=False
)


# ============================================================
# 23. SAVE EVERYTHING TO EXCEL
# ============================================================

print("\n")
print("Saving results to Excel...")


with pd.ExcelWriter(
    "mnist_gridsearch_results.xlsx",
    engine="openpyxl"
) as writer:

    activity1_results.to_excel(
        writer,
        sheet_name="Activity 1",
        index=False
    )

    activity2_results.to_excel(
        writer,
        sheet_name="Activity 2",
        index=False
    )

    activity3_results.to_excel(
        writer,
        sheet_name="Activity 3",
        index=False
    )

    final_results.to_excel(
        writer,
        sheet_name="Final Model",
        index=False
    )

    cm_dataframe.to_excel(
        writer,
        sheet_name="Confusion Matrix"
    )

    history_dataframe.to_excel(
        writer,
        sheet_name="Training History",
        index=False
    )


# ============================================================
# 24. FINAL SUMMARY
# ============================================================

print("\n")
print("#" * 70)
print("EXPERIMENT COMPLETE")
print("#" * 70)

print("\nBest Activity 3 configuration:")
print(
    f"Activation = {best_activation}"
)

print(
    f"Optimiser  = {best_optimiser}"
)

print(
    f"Neurons    = {best_neurons}"
)

print(
    f"CV accuracy = "
    f"{grid_activity3.best_score_ * 100:.2f}%"
)

print(
    f"Final test accuracy = "
    f"{test_accuracy * 100:.2f}%"
)

print("\nFiles saved:")

print(
    "activity1_gridsearch_results.csv"
)

print(
    "activity2_gridsearch_results.csv"
)

print(
    "activity3_gridsearch_results.csv"
)

print(
    "final_model_results.csv"
)

print(
    "confusion_matrix.csv"
)

print(
    "final_training_history.csv"
)

print(
    "mnist_gridsearch_results.xlsx"
)

print("\nDone.")


Loading MNIST dataset...

Dataset sizes:
Training: 54000
Validation: 6000
Test: 10000



######################################################################
ACTIVITY 1 - ACTIVATION FUNCTION GRID SEARCH
######################################################################
Fitting 2 folds for each of 4 candidates, totalling 8 fits
[CV] END model__activation=relu, model__neurons=10, model__optimiser=adam; total time=   9.3s
[CV] END model__activation=relu, model__neurons=10, model__optimiser=adam; total time=   9.0s
[CV] END model__activation=elu, model__neurons=10, model__optimiser=adam; total time=   8.8s
[CV] END model__activation=elu, model__neurons=10, model__optimiser=adam; total time=   8.8s
[CV] END model__activation=sigmoid, model__neurons=10, model__optimiser=adam; total time=   8.8s
[CV] END model__activation=sigmoid, model__neurons=10, model__optimiser=adam; total time=   9.8s
[CV] END model__activation=tanh, model__neurons=10, model__optimiser=adam; total time=   9.0s
[C